[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PerformanceEstimation/Tutorial-Europt-2026/blob/main/Part%203/Europt2026_Tutorial_Part3.ipynb)

# Third hands-on session with PEPit : difference-of-convex optimization

This notebook focuses on a third example: the difference-of-convex setup. In short, we want to minimize a potentially non-convex function that takes the form

$$\min_{x\in\mathbb{R}^d} \left\{f(x)\equiv f_1(x)-f_2(x)\right\}, $$

where both $f_1$ and $f_2$ are closed convex proper functions. We will mostly consider the cases where $f_1,f_2$ are continuously differentiable with Lipschitz gradients (smooth), and potentially strongly convex.

Below, we study the difference-of-convex algorithm (DCA), also known as the convex-concave procedure (CCP), described as follows: for $k \in \{ 0, \dots, n-1\}$

$$ x_{k+1} \in \mathrm{argmin}_{x\in\mathbb{R}^d}\left\{ f_1(x) - \langle \nabla f_2(x_k), x \rangle\right\}.$$

Note that optimality conditions of this subproblem yield $\nabla f_1(x_{k+1})=\nabla f_2(x_k)$.

## 1 Installing PEPit

In [ ]:
# If PEPit is not installed yet, you can run this cell.
!pip3 install pepit;

## 2 How to obtain a worst-case guarantee for the difference-of-convex algorithm (DCA) using PEPit

Similar to the previous examples with gradient descent, we are facing a smooth and potentially non-convex optimization problem, and our performance metric and initial conditions have to account for this. So, we will investigate perfomance bounds quantifying the capability of DCA to reach stationnary points. More precisely, we will study bounds of the type

$$ \min_{0\leq i\leq n} \|\nabla f(x_i)\|^2 \leq \tau_n (f(x_0)-f_\star),$$

where $f=f_1-f_2$ and $\nabla f=\nabla f_1-\nabla f_2$.

In this context, we describe the main four elements that allows PEPit to frame the analysis problem as an SDP:
- <font color='blue'>a class of problems</font> (containing the assumptions on the function to be minimized),
- <font color='red'>a first-order method</font> (to be analyzed),
- <font color='purple'>a performance measure</font> (measuring the quality of the output of the algorithm under consideration),
- <font color='green'>an initial condition</font> (measuring the quality of the initial iterate).

### 2.1 Imports

Before anything else, we have to import the PEP and the classes of functions of interest, as well as other necessary packages.

In [ ]:
# Import PEPit and the necessary class of functions
from PEPit import PEP
from PEPit.functions import SmoothStronglyConvexFunction

# Import the CVXPY namespace for solver selection
import cvxpy as cp

# Import numpy
import numpy as np

# Import matplotlib
import matplotlib.pyplot as plt

### 2.2 Initialization of PEPit
Then, we initialize a PEP object. This object allows manipulating the forthcoming ingredients of the PEP, such as functions and iterates.

In [ ]:
# Instantiate PEP
problem = PEP()

### 2.3 Choose parameters values

For the sake of the example, we pick some simple values for the problem class and algorithmic parameters, for which we perform the worst-case analysis below. For now, let us keep strong convexity parameters to 0.

In [ ]:
# Specify values for the smoothness and strong convexity parameters (mu_i, L_i), and number of iterations n.
L1 = 1.
L2 = 1.
mu1 = 0.
mu2 = 0.
n = 4

### 2.4 Specifying the problem class

Next, we specify our assumptions on the <font color='blue'>**class of functions**</font> containing the function to be optimized, and instantiate a corresponding object. Here, we consider the set of functions $f$ that can be written as a difference of smooth strongly convex functions with prescribed parameters.

In [ ]:
# Construct the objective function
f1 = problem.declare_function(SmoothStronglyConvexFunction, L=L1, mu=mu1)
f2 = problem.declare_function(SmoothStronglyConvexFunction, L=L2, mu=mu2)
F = f1 - f2

We also define $x_*$ as an optimal point and $f_*$ as the optimal value, as those are needed to express our performance criteria and initial condition.

In [ ]:
# Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
xs = F.stationary_point()
Fs = F(xs)

### 2.5 Initialization

Third, we can instantiate the starting point for the DCA that we will run, and specify an <font color='green'>**initial condition**</font> on this point.
To this end, a starting point $x_0$ is introduced, and a bound on the initial quality of $x_0$ is specified as $f(x_0)-f_*\leqslant 1$.

In [ ]:
# Then define the starting point x0 of the algorithm
x0 = problem.set_initial_point()

# Set the initial constraint
problem.set_initial_condition( F(x0) - Fs <= 1)

### 2.6 Setting up a performance measure

It is crucial for the worst-case analysis to specify the <font color='purple'>**performance metric**</font> for which we wish to compute a worst-case performance. In this example, we wish to compute the worst-case value of $\min_{0\leq i\leq n}\|\nabla f(x_i)\|^2$. By default, the way to study the **best** iterates in PEPit is to specify **several** performance metric (which will be put in epigraph form).

We start by specifying that at least $\|\nabla f(x_0)\|^2$ is a performance metric. We will then specify the ones for the other iterates when defining the algorithm update.

In [ ]:
# Set the performance metric to include \|\nabla F(x_0)\|^2
problem.set_performance_metric( F.gradient(x0)**2 )

### 2.7 Algorithm implementation

In this fourth step, we specify the <font color='red'>**algorithm**</font>. In this example, the algorithm is described as
$$ \nabla f_1(x_{k+1})=\nabla f_2(x_k). $$

We see that it would not yield the "most readable" algorithmic format if presented in a loop. Therefore, PEPit yields the possibility to implement *intermediate routines* to abstract the operation (known as [*primitive step*](https://pepit.readthedocs.io/en/latest/api/steps.html)). Below we implement the DCA step, which takes in input $x_k$, $f_1$, and $f_2$, and outputs and element of $\mathrm{argmin}_{x\in\mathbb{R}^d}\left\{ f_1(x) - \langle \nabla f_2(x_k), x \rangle\right\}$.

In [ ]:
from PEPit.point import Point
from PEPit.expression import Expression

def DCA_step(x, f1, f2):
    # Compute an output y of a DCA step with input (x,f1,f2). That is, y is such that
    # \nabla f_1(y) = \nabla f_2(x)

    # Get the gradient g2x and function value f2x of f2 in x
    g2x, f2x = f2.oracle(x)
    
    # Create two empty elements: a 'point' and an 'expression' to encode y and f1(y)
    y = Point()
    f1y = Expression()
    
    # Set the gradient of f1 at y to be equal to g2x
    g1y = g2x
    
    # Add (y,g1y,f1y) to the list of triplet to be interpolated for f1
    f1.add_point((y,g1y,f1y))

    # Return the newly obtained point
    return y

Run the algorithm for $n$ iterations; we also need to account for relationships between the function values at the iterates $F(x_i)$ and the optimal value $F(x_*)$ (see discussions in the [previous exercise session](https://github.com/PerformanceEstimation/Tutorial-Europt-2026/blob/main/Part%202/Europt2026_Tutorial_Part2.ipynb) for details).

In [ ]:
x = x0

# Upper bounds on F_*
problem.add_constraint(Fs <= F(x) - 1/2/(L1-mu2) * (F.gradient(x))**2)

# Run n steps of DCA
for _ in range(n):
    x = DCA_step(x,f1,f2)
    problem.set_performance_metric( F.gradient(x)**2 )
    problem.add_constraint(Fs <= F(x) - 1/2/(L1-mu2) * (F.gradient(x))**2)

### 2.8 Solving the PEP

The last natural stage in the process is to solve the corresponding PEP. This is done via the following line, which will ask PEPit to perform the modelling steps and to call an appropriate SDP solver (which should be installed beforehand) to perform the worst-case analysis.

Solving the SDP is handled by the [CVXPY](https://www.cvxpy.org/) modelling layer.

The default CVXPY solver [SCS](https://web.stanford.edu/~boyd/papers/scs.html)  may not be the most appropriate  for solving SDPs; we advise using [CLARABEL](https://clarabel.org/) (free, open-source, available in colab) or [MOSEK](https://www.mosek.com/) (commercial, free academic license) if possible, for improved numerical performances.

In [ ]:
# Solve the PEP with verbose = 1 and CLARABEL solver
pepit_tau = problem.solve(verbose=1, solver=cp.CLARABEL)
print(f"\n\n*** Performance bound of DCA after {n} steps is {pepit_tau}")

### 2.9 Create a Python routine and plotting worst-case guarantees as functions of number of steps

Complete the code below for a more convenient evaluation of PEP for different values of the parameters.

In [ ]:
def wc_DCA( mu1, L1, mu2, L2, n, verbose=1):

    # Instantiate PEP
    problem = PEP()
    
    # Construct the objective function
    f1 = problem.declare_function(SmoothStronglyConvexFunction, L=L1, mu=mu1)
    f2 = problem.declare_function(SmoothStronglyConvexFunction, L=L2, mu=mu2)
    F = f1 - f2

    # Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
    xs = F.stationary_point()
    Fs = F(xs)
    
    # Then define the starting point x0 of the algorithm
    x0 = problem.set_initial_point()
    
    # Set the initial constraint
    problem.set_initial_condition( F(x0) - Fs <= 1)
    
    # Set the performance metric to include \|\nabla F(x_0)\|^2
    problem.set_performance_metric( F.gradient(x0)**2 )
    
    # Prepare to run the algorithm
    x = x0
    # Upper bounds on F_*
    problem.add_constraint(Fs <= F(x) - 1/2/(L1-mu2) * (F.gradient(x))**2)
    
    # Run n steps of DCA
    for _ in range(n):
        x = DCA_step(x,f1,f2)
        problem.set_performance_metric( F.gradient(x)**2 )
        problem.add_constraint(Fs <= F(x) - 1/2/(L1-mu2) * (F.gradient(x))**2)
    
    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(verbose=pepit_verbose, solver=cp.CLARABEL)
    guessed_tau = 2 / ( 2 * n + 1 ) # YOUR GUESS HERE

    return pepit_tau, guessed_tau

In [ ]:
# Set the parameters
L1, L2 = 1, 1 # smoothness parameters
mu1, mu2 = 0., 0. # strong convexity parameters

n_max = 15   # max number of iterations to be studied
# Set a list of numner of steps
ns = range(1,n_max)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
guessed_taus = list()
for n in ns:
    pepit_tau, guessed_tau = wc_DCA( mu1=mu1, L1=L1, mu2=mu2, L2=L2, n=n, verbose=0 )
    pepit_taus.append(pepit_tau)
    guessed_taus.append(guessed_tau)
    print(f'{n+1} / {n_max} grid points computed', end='\r', flush=True)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
plt.plot(ns, guessed_taus, '--', label='Guessed tight bound')
plt.plot(ns, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps ($n$)')
plt.ylabel('Worst-case guarantee')

plt.show()

How could you have guessed the values of the worst-case guarantees as a function of $n$? 
> your turn!

Hint: plot the inverse of the taus again!

In [ ]:
inverse_pepit_taus = [1/tau for tau in pepit_taus]
plt.plot(ns, inverse_pepit_taus, 'x', label='Inverse PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps ($n$)')
plt.ylabel('Inverse worst-case guarantee')

plt.show()

## 3 The Boosted DCA

Now, let's consider a variation around DCA with an additional extrapolation (boosting) step of parameter $\alpha>0$: for $k \in \{ 0, \dots, n-1\}$

\begin{equation*}
\begin{aligned}
y_{k} &\in \mathrm{argmin}_{x\in\mathbb{R}^d}\left\{ f_1(x) - \langle \nabla f_2(x_k), x \rangle\right\}\\
d_{k+1} & = y_k - x_k \\
x_{k+1} & = y_k + \alpha d_k
\end{aligned}
\end{equation*}

Adapt the previous codes to account for this extrapolation step and experiment with the code. Can you find setups where $\alpha>0$ might help?

For simplicity, set $L_1=L_2=1$ and $\mu_1=\mu_2$.

In [ ]:
def wc_Boosted_DCA( mu1, L1, mu2, L2, alpha, n, verbose=1):

    # Instantiate PEP
    problem = PEP()
    
    # Construct the objective function
    f1 = problem.declare_function(SmoothStronglyConvexFunction, L=L1, mu=mu1)
    f2 = problem.declare_function(SmoothStronglyConvexFunction, L=L2, mu=mu2)
    F = f1 - f2

    # Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
    xs = F.stationary_point()
    Fs = F(xs)
    
    # Then define the starting point x0 of the algorithm
    x0 = problem.set_initial_point()
    
    # Set the initial constraint
    problem.set_initial_condition( F(x0) - Fs <= 1)
    
    # Set the performance metric to include \|\nabla F(x_0)\|^2
    problem.set_performance_metric( F.gradient(x0)**2 )
    
    # Prepare to run the algorithm
    x = x0
    # Upper bounds on F_*
    problem.add_constraint(Fs <= F(x) - 1/2/(L1-mu2) * (F.gradient(x))**2)
    
    # Run n steps of DCA
    for _ in range(n):
        y = DCA_step(x,f1,f2)
        d = y - x
        x = y + alpha * d
        problem.set_performance_metric( F.gradient(x)**2 )
        problem.add_constraint(Fs <= F(x) - 1/2/(L1-mu2) * (F.gradient(x))**2)
    
    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(verbose=pepit_verbose, solver=cp.CLARABEL)
    guessed_tau = 1/((1+mu1/L1*alpha)*n+1/2/(1-mu1/L1)) # If stepsize is small enough, this should be predictive.

    return pepit_tau, guessed_tau

### 3.1 Fix $n$ and plot as a function of boosting parameter $\alpha$

In [ ]:
# Set the parameters
L1, L2 = 1, 1 # smoothness parameters
mu1, mu2 = 0.1, 0.1 # strong convexity parameters

n = 3
nb_grid_elem = 20 # size of the grid of stepsizes
# Set a list of boosting parameters to test
alphas = np.linspace(.01, .45, nb_grid_elem)


# Compute numerical and theoretical (analytical) worst-case guarantees for the each alpha
pepit_taus_BDCA = list()
guessed_taus_BDCA = list()
for i in range(nb_grid_elem):
    alpha = alphas[i]
    pepit_tau, guessed_tau = wc_Boosted_DCA( mu1=mu1, L1=L1, mu2=mu2, L2=L2, alpha=alpha, n=n, verbose=0 )
    pepit_taus_BDCA.append(pepit_tau)
    guessed_taus_BDCA.append(guessed_tau)
    print(f'{i+1} / {nb_grid_elem} grid points computed', end='\r', flush=True)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
plt.plot(alphas, pepit_taus_BDCA, '--', label='PEPit bound (B-DCA)')
plt.plot(alphas, guessed_taus_BDCA, 'x', label='Guessed bound (B-DCA)')

plt.legend()
plt.xlabel(r'Boosting parameter ($\alpha$)')
plt.ylabel('Worst-case guarantee')

plt.show()

### 3.2 Fix the boosting parameter $\alpha$ and vary the iterations $n$

In [ ]:
# Set the parameters
L1, L2 = 1, 1 # smoothness parameters
mu1, mu2 = 0.2, 0.2 # strong convexity parameters
alpha = .4

n_max = 10   # max number of iterations to be studied
# Set a list of numner of steps
ns = range(1,n_max)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus_DCA = list()
guessed_taus_DCA = list()
pepit_taus_BDCA = list()
guessed_taus_BDCA = list()
for n in ns:
    pepit_tau, guessed_tau = wc_Boosted_DCA( mu1=mu1, L1=L1, mu2=mu2, L2=L2, alpha=alpha, n=n, verbose=0 )
    pepit_taus_BDCA.append(pepit_tau)
    guessed_taus_BDCA.append(guessed_tau)
    pepit_tau, guessed_tau = wc_DCA( mu1=mu1, L1=L1, mu2=mu2, L2=L2, n=n, verbose=0 )
    pepit_taus_DCA.append(pepit_tau)
    guessed_taus_DCA.append(guessed_tau)
    print(f'{n+1} / {n_max} grid points computed', end='\r', flush=True)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
plt.plot(ns, pepit_taus_DCA, '--', label='PEPit bound (DCA)')
plt.plot(ns, pepit_taus_BDCA, '--', label='PEPit bound (B-DCA)')

plt.legend()
plt.xlabel('Number of steps ($n$)')
plt.ylabel('Worst-case guarantee')

plt.show()

### Going further <a class="anchor" id="sec4"></a>

The base PEP analyses of DCA can be found in [1], then refined in [2]. Boosted variants were studied in [3].

More complete references and demos can be found in [PEPit's documentation and demos](https://github.com/PerformanceEstimation/PEPit/blob/master/ressources/demo).


---

### References

[1] H. Abbaszadehpeivasti, E. de Klerk, M. Zamani. *On the rate of convergence of the difference-of-convex algorithm (DCA)*, Journal of Optimization Theory and Applications, 2024. [https://doi.org/10.1007/s10957-023-02199-z](https://doi.org/10.1007/s10957-023-02199-z)

[2] T. Rotaru, P. Patrinos, F. Glineur. *Tight analysis of Difference-of-Convex Algorithm (DCA) improves convergence rates for Proximal Gradient Descent*, AISTATS, 2025. [arXiv:2503.04486](https://arxiv.org/pdf/2503.04486)

[3] H.Abbaszadehpeivasti, E. de Klerk, A. Taylor. *On the convergence rate of the boosted difference-of-convex algorithm (DCA)*, Optimization Letters (2026). [https://doi.org/10.1007/s11590-026-02305-w](https://doi.org/10.1007/s11590-026-02305-w)
